In [ ]:
# now that we have a pretrained model
# we will evaluate its pretraining performance

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

In [7]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from src.evals import evaluate_model, generate_and_print_sample
from src.dataloaderv1 import create_dataloaderV1
from src.gpt import GPT8TModel

from src.checkpointing import load_checkpoint

from src.utils import *
from src.config import config

from pathlib import Path
import tiktoken

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
path_to_checkpoint = Path.cwd().parent / "models" / "V0" / "checkpoints" / "latest_model.pt"
tokenizer = tiktoken.get_encoding('gpt2')

model = GPT8TModel(
    emb_dim = config['emb_dim'],
    vocab_size = config['vocab_size'],
    context_length = config['context_length'],
    n_layers = config['n_layers'],
    n_heads = config['n_heads'],
    emb_dropout = config['emb_dropout'],
    mha_dropout = config['mha_dropout'],
    dropout = config['dropout'],
    expanding_factor = 6
)
model.to(device)

model, _, _, _, tokens_seen, train_losses, _, val_losses, _, all_tokens_seen, best_val_loss = load_checkpoint(path_to_checkpoint, model, None, device)

In [14]:
print(model)

GPT8TModel(
  (tok_emb): Embedding(50257, 512)
  (pos_emb): Embedding(256, 512)
  (embeds_dropout): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (norm1): LayerNorm()
      (multiheadattention): MultiHeadAttention(
        (dropout): Dropout(p=0.1, inplace=False)
        (Wq): Linear(in_features=512, out_features=512, bias=False)
        (Wk): Linear(in_features=512, out_features=512, bias=False)
        (Wv): Linear(in_features=512, out_features=512, bias=False)
        (output_proj): Linear(in_features=512, out_features=512, bias=True)
      )
      (dropout): Dropout(p=0.1, inplace=False)
      (norm2): LayerNorm()
      (fnn): FeedForwardNetwork(
        (layers): Sequential(
          (0): Linear(in_features=512, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=512, bias=True)
        )
      )
    )
    (1): TransformerBlock(
      (norm1): LayerNorm()
      (multiheadattention): 

In [15]:
# now lets just use the model to make a simple prediction just to see using our own eyes how good the model is
# without any evaluations

# multiple text examples
sample_texts = ["I was running down the street when", "The little girl"]

for text in sample_texts:
    generate_and_print_sample(model, tokenizer, device, text, max_new_tokens=50, context_length=config['context_length'])

Generated text: I was running down the street when I was going to go to the park. I was going to the park and I was going to the park. I was going to the park and I was going to the park. I was going to the park and I was going to the park
Generated text: The little girl was so excited! She ran to the park and ran to the park.

When she got to the park, she saw a big tree with lots of leaves. She ran to the tree and ran to the tree. She saw a big tree


In [ ]:
# well as we can see the outputs are kinda not that bad for a 60Mill param model
# + we had lots of limitations that brought the model back in terms of training that we will cover later along the way
# for now lets plot the train losses and val losses